In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# --- BẢNG MASTER ---
products = pd.read_csv('dataset/products.csv')
customers = pd.read_csv('dataset/customers.csv', parse_dates=['signup_date'])
promotions = pd.read_csv('dataset/promotions.csv', parse_dates=['start_date', 'end_date'])
geography = pd.read_csv('dataset/geography.csv')

# --- BẢNG TRANSACTION ---
orders = pd.read_csv('dataset/orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv('dataset/order_items.csv')
payments = pd.read_csv('dataset/payments.csv')
shipments = pd.read_csv('dataset/shipments.csv', parse_dates=['ship_date', 'delivery_date'])
returns = pd.read_csv('dataset/returns.csv', parse_dates=['return_date'])
reviews = pd.read_csv('dataset/reviews.csv', parse_dates=['review_date'])

# --- BẢNG ANALYTICAL ---
sales = pd.read_csv('dataset/sales.csv', parse_dates=['Date'])
# File test và sample thường chỉ cần đọc bình thường
#sales_test = pd.read_csv('dataset/sales_test.csv') 
sample_submission = pd.read_csv('dataset/sample_submission.csv')

# --- BẢNG OPERATIONAL ---
inventory = pd.read_csv('dataset/inventory.csv', parse_dates=['snapshot_date'])
# Giả sử inventory_enhanced có cùng cột ngày với inventory
#inventory_enhanced = pd.read_csv('dataset/inventory_enhanced.csv', parse_dates=['snapshot_date'])
web_traffic = pd.read_csv('dataset/web_traffic.csv', parse_dates=['date'])

C:\Users\Admin\AppData\Local\Temp\ipykernel_23044\1409091705.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('dataset/order_items.csv')


In [3]:
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [4]:
order_counts = orders.groupby('customer_id')['order_id'].transform('count')

q1 = orders[['customer_id', 'order_date']]

In [5]:
q1 = q1.sort_values(['customer_id', 'order_date'])

In [6]:
q1['days_diff'] = q1.groupby('customer_id')['order_date'].diff().dt.days

In [7]:
q1['days_diff'].median() #?????????? C 

144.0

In [22]:
q2=products.copy()
q2['answer']=(products['price']-products['cogs'])/(products['price'])

In [ ]:
q2.groupby('segment')['answer'].mean().sort_values() # Q2 = D

segment
Everyday       0.236343
Trendy         0.240758
Balanced       0.258038
Performance    0.263650
Activewear     0.265600
All-weather    0.284176
Premium        0.285377
Standard       0.313442
Name: answer, dtype: float64

In [29]:
q3 = returns.merge(products[['product_id','category']], on='product_id',how='inner')

In [30]:
q3 = q3[q3['category']=='Streetwear']

In [ ]:
q3.groupby('return_reason')['product_id'].count().sort_values() #Q3 = B

return_reason
late_delivery       2159
changed_mind        3830
not_as_described    3854
defective           4330
wrong_size          7626
Name: product_id, dtype: int64

In [ ]:
web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values() #q4 = C

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

In [65]:
order_items['promo_id'].isnull().value_counts(normalize=True) #Q5 = C

promo_id
True     0.613365
False    0.386635
Name: proportion, dtype: float64

In [66]:
q6 = orders.groupby('customer_id')['order_id'].count().reset_index()

In [67]:
q6 = q6.merge(customers[['customer_id','age_group']], on='customer_id',how='inner')

In [69]:
q6 = q6.groupby('age_group').agg(
    ncus=('customer_id','count'),
    norder=('order_id','sum')
)

In [71]:
q6['rate'] = q6['norder']/q6['ncus']

In [ ]:
q6.sort_values(by='rate') #Q6 = A

,ncus,norder,rate
age_group,,,
18-24,12599,89057,7.068577
25-34,26802,190622,7.112230
35-44,23642,170368,7.206159
45-54,17193,124138,7.220264
55+,10010,72760,7.268731


In [74]:
sales['Revenue'].sum()

16430476585.529999

In [ ]:
payments['payment_value'].sum() #????????????

15680869265.429995

In [95]:
payments['payment_method'].unique()

array(['credit_card', 'cod', 'paypal', 'apple_pay', 'bank_transfer'],
      dtype=object)

In [ ]:
q7 = orders.merge(payments[['order_id','payment_value']], on='order_id',how='inner')
q7 = geography[['zip','region']].merge(q7[['zip','payment_value']], on='zip',how='inner')


In [ ]:
q7.groupby('region')['payment_value'].sum().sort_values() #Q7 = C

region
West       3.670227e+09
Central    4.719491e+09
East       7.291151e+09
Name: payment_value, dtype: float64

In [ ]:
q8 = orders[orders['order_status']=='cancelled'].groupby('payment_method')['order_id'].count()
q8.sort_values() #Q8 = A

payment_method
bank_transfer     2535
apple_pay         5190
paypal            7817
cod              15468
credit_card      28452
Name: order_id, dtype: int64

In [103]:
returns

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76
...,...,...,...,...,...,...,...
39934,RET-051470,832867,653,2022-12-27,wrong_size,3,24741.62
39935,RET-051471,832890,792,2022-12-30,late_delivery,1,560.50
39936,RET-051481,833005,449,2022-12-31,defective,1,10002.55
39937,RET-051494,833234,1085,2022-12-28,wrong_size,1,815.57


In [104]:
products

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406
...,...,...,...,...,...,...,...,...
2407,1260,VietMode MP-28,Casual,Activewear,S,red,4603.340000,2553.933032
2408,1261,VietMode MP-29,Casual,Activewear,M,black,5983.876433,4653.660702
2409,1262,VietMode MP-30,Casual,Activewear,L,orange,5983.876433,5684.682611
2410,1263,VietMode MP-31,Casual,Activewear,XL,blue,5984.370000,5685.151500


In [ ]:
q9 = returns.merge(products[['product_id','size']],on='product_id',how='inner')

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,size
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,M
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,L
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,XL
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,M
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,L
...,...,...,...,...,...,...,...,...
39934,RET-051470,832867,653,2022-12-27,wrong_size,3,24741.62,M
39935,RET-051471,832890,792,2022-12-30,late_delivery,1,560.50,S
39936,RET-051481,833005,449,2022-12-31,defective,1,10002.55,M
39937,RET-051494,833234,1085,2022-12-28,wrong_size,1,815.57,M


In [ ]:
return_counts = q9.groupby('size')['product_id'].count()

orders_with_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='inner')
order_counts = orders_with_size.groupby('size')['product_id'].count()

return_rate = return_counts / order_counts

return_rate.sort_values() #Q9 = A ?

size
XL    0.055200
M     0.055660
L     0.056250
S     0.056515
Name: product_id, dtype: float64

In [ ]:
q10 = payments.groupby('installments')['payment_value'].mean().sort_values()
q10 #q10 = C

installments
2       708.473729
1     24113.274166
12    24245.772694
3     24399.635486
6     24446.654403
Name: payment_value, dtype: float64